# Will it be an early Spring?

On February 2<sup>nd</sup> every year Punxsutawney Phil makes a prediction about if there will be an early Spring or if Winter will continue for 6 more weeks (till about mid-March). He is however not very accurate (well, according to [The Inner Circle](https://www.groundhog.org/inner-circle) he is 100% correct but the human handler may not interpret his response correctly). The overall goal is to be able to predict if it will be an early Spring.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help). To schedule an appointment, look at the client's Google Calendar and then reach out to them to propose a time. I estimate the first meeting will need about 20 minutes. Come prepared.

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
   * Must provide a function `generate_samples(years: list[int], phil_preds: list[bool]) -> pd.DataFrame` that takes a list of years, a list of `True`/`False`s for Phil's prediction for those years, and gives back a data frame for those years matching the formatting of the train and test sets.
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes a the train or test set (or anything returned by `generate_samples()`) and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preperation should be noted here but actually implemented in the Preperation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit all data files and a pickled final model along with this notebook.

# Frame the Problem and Look at the Big Picture

1. **Define the objective in business terms.** 
We wish to figure out if the upcoming year will be an early spring.
2. **How will your solution be used?** 
This will allow farmers the ability to plant a new seed variant of corn that could yield two harvests if planted early given an early spring.
3. **What are the current solutions/workarounds (if any)?** 
Every year on Groundhog day, a groundhog is observed to see if it comes out of hibernation. If he, Puxatawny Phil, decides to come out, Spring is said to come early. If he goes back into his hole, it is predicted that Winter will continue for longer. He is right ~39% of the time (not very good).
4. **How should you frame this problem (supervised/unsupervised, online/offline, …)?** 
supervised, classification, offline: since it is only needed once a year.
5. **How should performance be measured? Is the performance measure aligned with the business objective?** 
The price of the seeds allow for us to be wrong sometimes. So we can be a little generous with how often we predict early Spring. Based on the price of the seeds, we can be wrong 5 times before profits are ruined.
6. **What would be the minimum performance needed to reach the business objective?** 
Like mentioned above, we can be a little generous with how often we predict early spring, so we should prioritize recall over precision.
7. **What are comparable problems? Can you reuse experience or tools?** 
One comparable problem is the demand in certain areas for bike gathering. It too was a supervised classification problem and it measured the demand through a few categories from low to high demand. We can reuse some of the strategies in this previous problem to expedite the completion of our current early spring ML problem.
8. **Is human expertise available?** 
Yes, Farmer Jeff and Professor Jeff are available as human expertise. Farmer Jeff will provide us with any information we need regarding the business problem and how we should approach a solution that will be beneficial to farmers. Professor Jeff is our mentor and can help us with any issues or parts of the process we are unsure about.
9. **How would you solve the problem manually?** 
So far, they solve the problem manually by looking to a Groundhog and seeing if he stays out of his hole. If stays, early spring, if not late spring.
10. **List the assumptions you (or others) have made so far. Verify assumptions if possible.** 
We can assume that we are able to be a little lenient towards predicting early spring to give the farmers a better chance at turning a profit. Since this strain of seed would allow two harvests if planted early enough.

# Get the Data

In [5]:
import io
import os
import json
import urllib.request
import numpy as np
import pandas as pd

In [6]:
def download(url, filename=None):
    """
    Downloads a file from the internet if it does not already exist locally
    """
    if filename is None:
        filename = os.path.basename(url)
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
    return filename


# Client-provided data (Farmer Jeff)
def read_early_spring():
    """Label: one row per year, True if that year had an early spring."""
    return pd.read_csv("early_spring.csv").sort_values("year").reset_index(drop=True)

def read_phil_pred():
    """Punxsutawney Phil's prediction for each year (True = early spring)."""
    return pd.read_csv("phil_pred.csv")

def read_farm_weather():
    """
    Daily farm weather (Jan-Mar of each year).
    The file uses NOAA-style codes inside numeric columns:
      'M' = missing  -> NaN
      'T' = trace amount (precip/snow too small to measure) -> 0.0
    """
    weather = pd.read_csv("weather.csv", parse_dates=["date"], na_values=["M"])
    value_cols = weather.columns.drop("date")
    weather[value_cols] = weather[value_cols].replace("T", 0.0).astype(float)
    return weather

def read_farm_soil():
    """Daily farm soil temperature/moisture (Jan-Mar of each year)."""
    return pd.read_csv("soil.csv", parse_dates=["day"])

Farm weather is coming from the files provided by Farmer Jeff, we also took the liberty of gathering Open-meteo data that contains daily historical weather analysis to flesh out any weather related information throughout the months of Jan and Feb. We also gathered ONI data that looks at El Nino and La Nina.

In [7]:
# Additional data sources
PUNXSUTAWNEY_LAT, PUNXSUTAWNEY_LON = 40.9437, -78.9709

def read_open_meteo(start_year=1947, end_year=2026):
    """
    Daily historical weather reanalysis (ERA5, back to 1940) for Punxsutawney, PA
    from the Open-Meteo Historical Weather API (free, no API key).
    Source: https://open-meteo.com/en/docs/historical-weather-api
    Only Jan 1 - Mar 31 is kept to match the farm data window.
    """
    name = f"open_meteo_{start_year}-{end_year}.json"
    if not os.path.exists(name):
        url = (
            "https://archive-api.open-meteo.com/v1/archive"
            f"?latitude={PUNXSUTAWNEY_LAT}&longitude={PUNXSUTAWNEY_LON}"
            f"&start_date={start_year}-01-01&end_date={end_year}-08-31"
            "&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min,"
            "precipitation_sum,snowfall_sum,sunshine_duration"
            "&timezone=America/New_York"
        )
        download(url, name)
    with open(name) as f:
        raw = json.load(f)
    om = pd.DataFrame(raw["daily"])
    om["time"] = pd.to_datetime(om["time"])
    om = om[om["time"].dt.month <= 3].reset_index(drop=True)
    return om


def read_psl_index(index_name):
    """
    Monthly climate index from NOAA PSL in their standard text format:
      first line  = "<first_year> <last_year>"
      then rows   = "<year> <jan> <feb> ... <dec>"
      then footer = missing-value marker + description lines
    index_name examples: "oni" (El Nino / ENSO, from 1950), "nao" (North Atlantic Oscillation).
    Source: https://psl.noaa.gov/data/climateindices/list/
    Returns a long table: year, month, value.
    """
    name = download(f"https://psl.noaa.gov/data/correlation/{index_name}.data", f"{index_name}.data")
    with open(name) as f:
        lines = f.read().splitlines()
    first_year, last_year = map(int, lines[0].split()[:2])
    n_years = last_year - first_year + 1
    missing_marker = float(lines[1 + n_years].split()[0]) 

    table = pd.read_csv(io.StringIO("\n".join(lines[1:1 + n_years])), sep=r"\s+", header=None)
    table.columns = ["year"] + list(range(1, 13))
    long = table.melt(id_vars="year", var_name="month", value_name=index_name)
    long[index_name] = long[index_name].replace(missing_marker, np.nan)
    return long.sort_values(["year", "month"]).reset_index(drop=True)

Like mentioned in the cell above, Open Meteo Data was gathered, where we got temperature and precipitation data, as well as snowfall and snowdepth for Punxsutawney, PA. For the PSL data, this is the ONI we were speaking of, it is a monthly climate index where we specifically took out ONI data which is El Nino and La Nina. It will potentially be helpful because it signals cold fronts and is what a human would most likely refer to if they were to make the prediction for themselves.

In [8]:
early_spring = read_early_spring()
phil_pred = read_phil_pred()
farm_weather = read_farm_weather()
farm_soil = read_farm_soil()
open_meteo = read_open_meteo()
oni = read_psl_index("oni")

for label, df in [("early_spring", early_spring), ("phil_pred", phil_pred),
                  ("farm_weather", farm_weather), ("farm_soil", farm_soil),
                  ("open_meteo", open_meteo), ("oni", oni)]:
    print(f"{label:13s} {df.shape}")

early_spring  (72, 2)
phil_pred     (72, 2)
farm_weather  (4338, 7)
farm_soil     (4338, 7)
open_meteo    (7220, 7)
oni           (924, 3)


In [9]:
# Turning daily data into one row per year

FREEZING_F = 32.0  # farm weather temps are in Fahrenheit
FREEZING_C = 0.0   # soil and Open-Meteo temps are in Celsius


def _window_and_period(df, date_col):
    """Keep only rows in the Jan 1 - Mar 1 window and tag each with its year and period."""
    dates = df[date_col]
    in_window = (dates.dt.month <= 2) | ((dates.dt.month == 3) & (dates.dt.day == 1))
    df = df[in_window].copy()
    df["year"] = df[date_col].dt.year
    df["period"] = np.where(df[date_col].dt.month == 1, "jan", "feb")
    return df


def _per_year(df, prefix, aggregations):
    """
    Group by (year, period), apply the named aggregations, then flatten so each
    year is one row with columns like "<prefix>_<feature>_<period>".
    aggregations: dict of output_name -> (column, aggfunc), same as pandas named agg.
    """
    stats = df.groupby(["year", "period"]).agg(**aggregations).unstack("period")
    stats.columns = [f"{prefix}_{feature}_{period}" for feature, period in stats.columns]
    return stats

def _last_7d_features(df, date_col, prefix, value_cols):
    """
    Extracts daily features for the 7 days leading up to March 1st (d_minus_1 to d_minus_7)
    and relative summary stats, naturally adjusting for leap years.
    """
    df = df.copy()
    dates = pd.to_datetime(df[date_col])

    # Construct March 1st of each row's year as the anchor date
    march_1st = pd.to_datetime(dates.dt.year.astype(str) + "-03-01")

    # Offset in days: 1 day before Mar 1 is d_minus_1, 7 days before is d_minus_7
    days_before = (march_1st - dates).dt.days

    # Filter to exact 7-day window
    in_7d = (days_before >= 1) & (days_before <= 7)
    sub = df[in_7d].copy()

    sub["year"] = dates[in_7d].dt.year
    sub["lag"] = "d_minus_" + days_before[in_7d].astype(str)

    # Ensure value_cols is a list
    if isinstance(value_cols, str):
        value_cols = [value_cols]

    # 1. Individual daily lag features (pivot)
    pivot = sub.pivot(index="year", columns="lag", values=value_cols)
    pivot.columns = [f"{prefix}_{col}_{lag}" for col, lag in pivot.columns]

    # 2. Aggregated 7-day stats (mean)
    means = sub.groupby("year")[value_cols].mean()
    means.columns = [f"{prefix}_{col}_last_7d_mean" for col in means.columns]

    return pivot.join(means, how="left")


def _farm_weather_features(weather):
    weather_win = _window_and_period(weather, "date")
    weather_win["below_freezing"] = weather_win["min_temp"] <= FREEZING_F
    weather_win["snow_on_ground"] = weather_win["snowdepth"] > 0

    # Jan/Feb monthly aggregations
    monthly_stats = _per_year(
        weather_win,
        "farm",
        {
            "avg_temp_mean": ("avg_temp", "mean"),
            "min_temp_min": ("min_temp", "min"),
            "max_temp_max": ("max_temp", "max"),
            "precip_sum": ("precipitation", "sum"),
            "snowfall_sum": ("snowfall", "sum"),
            "snowdepth_mean": ("snowdepth", "mean"),
            "freezing_days": ("below_freezing", "sum"),
            "snow_cover_days": ("snow_on_ground", "sum"),
        },
    )

    # 7-day pre-March features for key daily variables
    last_7_stats = _last_7d_features(
        weather,
        date_col="date",
        prefix="farm",
        value_cols=["avg_temp", "snowdepth"],
    )

    return monthly_stats.join(last_7_stats, how="left")


def _farm_soil_features(soil):
    soil_win = _window_and_period(soil, "day")
    soil_win["soil_above_freezing"] = soil_win["mean_soil_temp"] > FREEZING_C

    monthly_stats = _per_year(
        soil_win,
        "soil",
        {
            "temp_mean": ("mean_soil_temp", "mean"),
            "temp_min": ("min_soil_temp", "min"),
            "temp_max": ("max_soil_temp", "max"),
            "moisture_mean": ("mean_soil_moisture", "mean"),
            "thawed_days": ("soil_above_freezing", "sum"),
        },
    )

    last_7_stats = _last_7d_features(
        soil,
        date_col="day",
        prefix="soil",
        value_cols=["mean_soil_temp", "mean_soil_moisture"],
    )

    return monthly_stats.join(last_7_stats, how="left")


def _open_meteo_features(om):
    om_win = _window_and_period(om, "time")
    om_win["below_freezing"] = om_win["temperature_2m_min"] <= FREEZING_C
    om_win["sunshine_hours"] = om_win["sunshine_duration"] / 3600

    monthly_stats = _per_year(
        om_win,
        "om",
        {
            "temp_mean": ("temperature_2m_mean", "mean"),
            "temp_min": ("temperature_2m_min", "min"),
            "temp_max": ("temperature_2m_max", "max"),
            "precip_sum": ("precipitation_sum", "sum"),
            "snowfall_sum": ("snowfall_sum", "sum"),
            "sunshine_hours_sum": ("sunshine_hours", "sum"),
            "freezing_days": ("below_freezing", "sum"),
        },
    )

    last_7_stats = _last_7d_features(
        om_win,  # om already has sunshine_hours calculated
        date_col="time",
        prefix="om",
        value_cols=["temperature_2m_mean", "precipitation_sum"],
    )

    return monthly_stats.join(last_7_stats, how="left")


def _oni_features(oni):
    """
    ONI values are 3-month running means labeled by their center month, so the
    January value (DJF) needs February data and isn't published until March.
    To stay inside the Mar 1 cutoff, use the previous year's Nov (OND) and Dec (NDJ).
    """
    prev = oni[oni["month"].isin([11, 12])].copy()
    prev["year"] = prev["year"] + 1  # previous Nov/Dec belongs to the next spring
    wide = prev.pivot(index="year", columns="month", values="oni")
    wide.columns = [f"oni_prev_{'nov' if m == 11 else 'dec'}" for m in wide.columns]
    return wide


def generate_samples(years: list[int], phil_preds: list[bool]) -> pd.DataFrame:
    """
    Build one row per year with Phil's prediction plus Jan 1 - Mar 1 features from
    every source. Years missing from a source get NaN for that source's columns.
    The early_spring label is filled in when it is known (NaN otherwise) so the
    output has the same columns as train_set / test_set.
    """
    if len(years) != len(phil_preds):
        raise ValueError("years and phil_preds must be the same length")

    samples = pd.DataFrame({"phil_pred": phil_preds}, index=pd.Index(years, name="year"))
    for features in (_farm_weather_features(farm_weather),
                     _farm_soil_features(farm_soil),
                     _open_meteo_features(open_meteo),
                     _oni_features(oni)):
        samples = samples.join(features, how="left")

    labels = early_spring.set_index("year")["early_spring"]
    samples["early_spring"] = labels.reindex(samples.index)
    return samples

We take all the data sources we have collected, and pool them together to get by year data. We extract the 7 days before march 1st and also form aggregations for the months statistics in January and February for each year. We also took the ONI data and extracted the nov and dec from the previous year, since March 1st falls before those months.

In [10]:
# Split the train and test datasets
from sklearn.model_selection import train_test_split

phil_by_year = phil_pred.set_index("year")["prediction"].astype(bool)
labeled_years = early_spring["year"].tolist()
full = generate_samples(labeled_years, phil_by_year.loc[labeled_years].tolist())
full["early_spring"] = full["early_spring"].astype(bool)

train_set, test_set = train_test_split(
    full, test_size=0.2, random_state=42, stratify=full["early_spring"]
)

print(train_set.shape, test_set.shape)
train_set.to_parquet("train_set.parquet")
test_set.to_parquet("test_set.parquet")

(57, 92) (15, 92)


We decided to stratify on our target because it is heavily skewed, so it would require proper division in order to get adequate train/test sets. We also decided to hold onto Puxatawny Phil's predictions for comparison later when we train our models. It is a fun little comparison we can do and a helpful baseline we can try to beat (hopefully).